In [ ]:
!git clone -b main_hayden https://github.com/prince-modi/gpu-kernel-dev.git
%cd gpu-kernel-dev

In [ ]:
!pip install -r requirements.txt
!pip install triton torch helion

In [3]:
import os
import shutil
from google.colab import files

# was running into issue with CUTE so temp-fix set the arch to sm_80 for t4
os.environ["CUTE_DSL_ARCH"] = "sm_80"

triton_cache = os.path.expanduser("~/.triton/cache")
if os.path.exists(triton_cache):
    shutil.rmtree(triton_cache)

!python3 bench-driver.py --generate-kernel-dump=rms_bench-triton-rmsnorm_with_loops --M 1024 --N 1024

if os.path.exists(triton_cache):
    zip_path = "/content/t4_triton_dump"
    shutil.make_archive(zip_path, 'zip', triton_cache)
    print(f"Packaged into {zip_path}.zip")

    files.download(f"{zip_path}.zip")
else:
    print("\nERROR: No cache was found.")

CUDA is available. Using GPU.
CUDA is available. Using GPU.
Success! Packaged into /content/t4_triton_dump.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
triton_cache = os.path.expanduser("~/.triton/cache")
if os.path.exists(triton_cache):
    shutil.rmtree(triton_cache)

os.environ["CUTE_DSL_ARCH"] = "sm_80"

!python3 bench-driver.py --generate-kernel-dump=attn_bench-triton-forward

if os.path.exists(triton_cache):
    zip_path = "/content/t4_flash_attn_dump"
    shutil.make_archive(zip_path, 'zip', triton_cache)
    print(f"Packaged into {zip_path}.zip")

    files.download(f"{zip_path}.zip")
else:
    print("ERROR: No cache was found.")

Old cache cleared. Ready for fresh compilation.

Running bench-driver.py for isolated compilation...
CUDA is available. Using GPU.
CUDA is available. Using GPU.
Traceback (most recent call last):
  File "/content/gpu-kernel-dev/bench-driver.py", line 44, in <module>
    method()
  File "/content/gpu-kernel-dev/attn_benchmark_driver.py", line 116, in <lambda>
    return lambda: tlb.attn_benchmarks(bench_name, Q=q, K=k, V=v)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/gpu-kernel-dev/triton_kernels/benchmark.py", line 57, in attn_benchmarks
    return flash_attention_v2_wrapper(q, k, v, causal=False)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/gpu-kernel-dev/triton_kernels/flashattn/flash_attn_v2.py", line 183, in flash_attention_v2_wrapper
    attn_fwd[grid](
  File "/usr/local/lib/python3.12/dist-packages/triton/runtime/jit.py", line 370, in <lambda>
    return lambda *args, **kwargs: self.run(grid=grid, warmup=Fal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import os
import shutil
import json

os.environ["CUTE_DSL_ARCH"] = "sm_80"
os.environ["HELION_SKIP_AUTOTUNE"] = "1"
os.environ["TRITON_CACHE_DIR"] = os.path.expanduser("~/.triton/cache")
os.environ["TRITON_STORE_BINARY_ONLY"] = "0"  # Force saving .ptx/.ttgir
os.environ["HELION_KEEP_TRITON_CACHE"] = "1" # Prevent Helion from cleaning up

triton_cache = os.environ["TRITON_CACHE_DIR"]
if os.path.exists(triton_cache):
    shutil.rmtree(triton_cache)
    print("Cache cleared.")

os.makedirs("configs", exist_ok=True)

rms_config_1 = {"block_sizes": [1024], "num_warps": 4, "num_stages": 2}
rms_config_2 = {"block_sizes": [512],  "num_warps": 4, "num_stages": 2}

with open("configs/helion_rms_kernel-a.json", "w") as f:
    json.dump(rms_config_1, f)
with open("configs/helion_rms_kernel-b.json", "w") as f:
    json.dump(rms_config_2, f)

fa_config_1 = {"block_sizes": [1, 64, 64], "num_warps": 4, "num_stages": 2}
fa_config_2 = {"block_sizes": [1, 32, 32], "num_warps": 4, "num_stages": 2}

with open("configs/flashatt_fwd-a.json", "w") as f:
    json.dump(fa_config_1, f)
with open("configs/flashatt_fwd-b.json", "w") as f:
    json.dump(fa_config_2, f)

!TRITON_PRINT_AUTOTUNING=1 TRITON_DEBUG=1 python3 bench-driver.py \
    --generate-kernel-dump=rms_bench-helion-helion_rms_kernel \
    --M 1024 --N 1024

!TRITON_PRINT_AUTOTUNING=1 TRITON_DEBUG=1 python3 bench-driver.py \
    --generate-kernel-dump=attn_bench-helion-flashatt_fwd

if os.path.exists(triton_cache):
    if any(os.scandir(triton_cache)):
        zip_path = "/content/helion_kernel_dumps"
        shutil.make_archive(zip_path, 'zip', triton_cache)
        print(f"Helion artifacts packaged into {zip_path}.zip")
        from google.colab import files
        files.download(f"{zip_path}.zip")
    else:
        print("\nERROR: Cache folder exists but is EMPTY. Triton did not write IR files.")
else:
    print("\nERROR: No cache directory created at all.")

CUDA is available. Using GPU.
CUDA is available. Using GPU.
CUDA is available. Using GPU.
CUDA is available. Using GPU.
Success! Helion artifacts packaged into /content/helion_kernel_dumps.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>